# 1.4 计算与通信调优

## 本节学习目标

- 理解 MGS 主线与 CGS 运行时能力探测
- 控制 collective 与线程预算

## 环境检查

直接检查本节需要的运行环境；若检查失败，请先在对应 CPU/NPU 节点加载课程要求的工具链。


In [ ]:
%%bash
set -e
command -v cmake
command -v msprof
command -v npu-smi
npu-smi info
printf "ASCEND_HOME_PATH=%s\n" "${ASCEND_HOME_PATH:?请先 source CANN set_env.sh}"


## 正交化

MGS 每加入一个基向量通常触发标量 AllReduce；communication-avoiding CGS 的设计目标是把同一步多个系数合并为一次向量 AllReduce，以减少调用次数。当前仓库的 Device 源码明确拒绝 CGS（`Device CGS multi-dot kernel is not enabled`）；实验仍按运行结果做能力探测，避免把不同源码或旧二进制误判为同一能力。

## 计算后端与线程预算

正式路径中 SpMV/Dot/Norm/AXPY/Scale 由 Ascend C RTC Device kernel 执行（`kernels/gmres_ops.cpp`）；仅无 CANN 的 Host stub 路径使用 OpenMP reference。多 rank 同时使用过多 CPU 线程仍会过度订阅，脚本按 host CPU 数限制每 rank 线程，并提供固定总线程预算。

## 查看优化开关

从当前章节目录执行下面的 Cell，并对照随后给出的检查点阅读输出。

## 预期现象与结果分析

CGS 的设计目标是减少 collective，但不保证总时间必然下降。若能力探测成功，必须同时检查数值稳定性、迭代数、local compute 和同步等待；若明确不支持，则记录拒绝信息，不把它写成性能对照。

## 原工程优化前后对照

原工程 README 使用相同卡数下的 Distributed total 比较调优前后，避免旧版 single baseline 抖动扭曲结论。代表性记录如下：

<table style="margin-left: 0; text-align: left;">
  <thead>
    <tr>
      <th>Matrix</th>
      <th>Rank</th>
      <th>优化前 (ms)</th>
      <th>优化后 (ms)</th>
      <th>端到端缩短</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>U1</td>
      <td>4</td>
      <td>697.988</td>
      <td>16.292</td>
      <td>42.84x</td>
    </tr>
    <tr>
      <td>U2</td>
      <td>8</td>
      <td>1646.913</td>
      <td>126.988</td>
      <td>12.97x</td>
    </tr>
    <tr>
      <td>L2</td>
      <td>8</td>
      <td>18574.357</td>
      <td>514.266</td>
      <td>36.12x</td>
    </tr>
    <tr>
      <td>B2</td>
      <td>8</td>
      <td>6989.999</td>
      <td>199.689</td>
      <td>35.00x</td>
    </tr>
  </tbody>
</table>

主要改善来自限制每 rank OpenMP 线程、短向量串行阈值、融合向量循环、减少 collective，以及复用 buffer；数学迭代次数没有改变。这组历史数据适合演示“profile → 假设 → 单变量验证”的方法，不能归因为 AI Core Kernel 优化。

## 课后实践

固定其他参数，设计 partition rows|nnz 的单变量对照（CGS 先做能力探测；OpenMP 开关只影响 Host stub 路径）。

参考答案见 `answer/01.04_answer.md`。